# Phase 7: Transferability Profile & Prediction

Goals:
- combine all transferability dimensions
- compute the overall transferability score
- estimate the probability of DA success
- final recommendations

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Path settings
ROOT = r'..'
BENCHMARK_DIR = os.path.join(ROOT, 'benchmark')
RESULTS_DIR = os.path.join(BENCHMARK_DIR, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

print("="*100)
print("PHASE 7: Transferability Profile & Prediction")
print("="*100)
print(f"Results: {RESULTS_DIR}\n")

PHASE 7: Transferability Profile & Prediction
Results: ..\benchmark\results



In [2]:
# Load all phase results
print("Loading all phase results...\n")

with open(os.path.join(RESULTS_DIR, 'phase_01_overlap.json'), 'r') as f:
    phase_01 = json.load(f)

with open(os.path.join(RESULTS_DIR, 'phase_02_inter_similarity.json'), 'r') as f:
    phase_02 = json.load(f)

with open(os.path.join(RESULTS_DIR, 'phase_03_representation_gap.json'), 'r') as f:
    phase_03 = json.load(f)

with open(os.path.join(RESULTS_DIR, 'phase_04_label_correspondence.json'), 'r') as f:
    phase_04 = json.load(f)

with open(os.path.join(RESULTS_DIR, 'phase_05_domain_discriminability.json'), 'r') as f:
    phase_05 = json.load(f)

with open(os.path.join(RESULTS_DIR, 'phase_06_dataset_compatibility.json'), 'r') as f:
    phase_06 = json.load(f)

print("✓ All phases loaded")
print(f"  Phases: 1, 2, 3, 4, 5, 6\n")

Loading all phase results...

✓ All phases loaded
  Phases: 1, 2, 3, 4, 5, 6



In [3]:
def normalize_score(value, min_val, max_val, invert=False):
    """
    Normalize value to 0-100 score
    invert: if True, higher value -> lower score (for distances)
    """
    if value is None:
        return None
    
    normalized = (value - min_val) / (max_val - min_val + 1e-10)
    normalized = np.clip(normalized, 0, 1)
    
    if invert:
        normalized = 1 - normalized
    
    return int(100 * normalized)

# Transferability Profile Calculation
transferability_profiles = {}

properties = list(phase_01.keys())

for prop in properties:
    print(f"\n{'='*80}")
    print(f"Property: {prop.upper()}")
    print(f"{'='*80}")
    
    profile = {}
    scores = {}
    
    # Dimension 1: Chemical Similarity (Phase 2)
    # Inter-domain similarity: higher is better
    if prop in phase_02 and 'mean' in phase_02[prop]:
        chem_sim = phase_02[prop]['mean']
        # Range: 0-0.5 typical for different domains, 0.5+ for similar
        chem_score = normalize_score(chem_sim, 0, 0.5, invert=False)
        profile['chemical_similarity'] = float(chem_sim)
        scores['Chemical Similarity'] = chem_score
        print(f"  1. Chemical Similarity: {chem_sim:.4f} → Score {chem_score}/100")
    
    # Dimension 2: Representation Gap (Phase 3)
    # MMD distance: lower is better (invert=True)
    if prop in phase_03 and 'mmd_distance' in phase_03[prop]:
        mmd = phase_03[prop]['mmd_distance']
        # Typical range: 0-1.0
        rep_score = normalize_score(mmd, 0, 1.0, invert=True)
        profile['representation_gap_mmd'] = float(mmd)
        scores['Representation Gap'] = rep_score
        print(f"  2. Representation Gap (MMD): {mmd:.4f} → Score {rep_score}/100")
    
    # Dimension 3: Label Correspondence (Phase 4)
    # Pearson correlation: higher is better
    if prop in phase_04 and phase_04[prop].get('spearman_rho') is not None:
        label_r = phase_04[prop]['spearman_rho']  # 1-NN (Tanimoto ≥ 0.4) in vitro↔in vivo label correlation
        # Range: -1 to 1, but we expect positive
        label_score = normalize_score(label_r, 0, 1, invert=False)
        profile['label_correlation'] = float(label_r)
        scores['Label Correspondence'] = label_score
        print(f"  3. Label Correspondence (r): {label_r:.4f} → Score {label_score}/100")
    else:
        # No overlap
        label_score = 50  # Neutral when no overlap
        scores['Label Correspondence'] = label_score
        print(f"  3. Label Correspondence: No overlap → Score {label_score}/100 (neutral)")
    
    # Dimension 4: Domain Discriminability (Phase 5)
    # AUC: 0.5=similar (good), 1.0=different (bad), invert=True
    if prop in phase_05 and 'avg_auc' in phase_05[prop]:
        auc = phase_05[prop]['avg_auc']
        # AUC 0.5 is best (score 100), AUC 1.0 is worst (score 0)
        domain_score = normalize_score(auc, 1.0, 0.5, invert=False)
        domain_score = int(np.clip(domain_score, 0, 100))
        profile['domain_auc'] = float(auc)
        scores['Domain Discriminability'] = domain_score
        print(f"  4. Domain Discriminability (AUC): {auc:.4f} → Score {domain_score}/100")
    
    # Dimension 5: Dataset Compatibility (Phase 6)
    if prop in phase_06 and 'compatibility_score' in phase_06[prop]:
        compat_score = phase_06[prop]['compatibility_score']
        profile['dataset_compatibility'] = int(compat_score)
        scores['Dataset Compatibility'] = compat_score
        print(f"  5. Dataset Compatibility: {compat_score}/100")
    
    # Dimension 6: Overlap Prevalence (Phase 1)
    if prop in phase_01 and 'overlap_count' in phase_01[prop]:
        overlap_count = phase_01[prop]['overlap_count']
        overlap_pct = phase_01[prop].get('overlap_pct_src', 0)
        
        # Overlap is good for alignment, score based on percentage
        # Ideal: 5-30% overlap (enough for alignment, not too much)
        if overlap_pct > 30:
            overlap_score = 80  # Good
        elif overlap_pct > 10:
            overlap_score = 90  # Excellent
        elif overlap_pct > 1:
            overlap_score = 60  # Moderate
        else:
            overlap_score = 30  # Poor
        
        profile['overlap_count'] = int(overlap_count)
        profile['overlap_pct'] = float(overlap_pct)
        # Overlap was removed when building the CV splits (0 by design), so it is excluded from the score
        print(f"  6. Overlap Prevalence: {overlap_pct:.1f}% ({overlap_count} molecules) → Score {overlap_score}/100")
    
    # Overall Transferability Score (weighted average)
    if scores:
        # Weights (can be adjusted based on empirical findings)
        weights = {
            'Chemical Similarity': 0.15,
            'Representation Gap': 0.20,
            'Label Correspondence': 0.20,
            'Domain Discriminability': 0.20,
            'Dataset Compatibility': 0.15,
            'Overlap Prevalence': 0.10,
        }
        
        overall_score = 0
        weight_sum = 0
        for dim, score in scores.items():
            w = weights.get(dim, 0)
            overall_score += score * w
            weight_sum += w
        
        overall_score = int(overall_score / weight_sum) if weight_sum > 0 else 0
        
        print(f"\n  Overall Transferability Score: {overall_score}/100")
        
        # DA Success Probability Estimation
        if overall_score >= 80:
            da_success_prob = 0.85
            recommendation = "STRONGLY RECOMMENDED"
            confidence = "High"
        elif overall_score >= 70:
            da_success_prob = 0.70
            recommendation = "RECOMMENDED"
            confidence = "High"
        elif overall_score >= 60:
            da_success_prob = 0.55
            recommendation = "WORTH TRYING"
            confidence = "Medium"
        elif overall_score >= 50:
            da_success_prob = 0.40
            recommendation = "MARGINAL"
            confidence = "Medium"
        else:
            da_success_prob = 0.20
            recommendation = "NOT RECOMMENDED"
            confidence = "Low"
        
        print(f"  DA Success Probability: {da_success_prob:.0%}")
        print(f"  Recommendation: {recommendation}")
        print(f"  Confidence: {confidence}")
        
        profile['overall_score'] = overall_score
        profile['da_success_probability'] = float(da_success_prob)
        profile['recommendation'] = recommendation
        profile['confidence'] = confidence
        profile['individual_scores'] = scores
        
        transferability_profiles[prop] = profile


Property: CLEARANCE
  1. Chemical Similarity: 0.0947 → Score 18/100
  2. Representation Gap (MMD): 0.0390 → Score 96/100
  3. Label Correspondence (r): 0.1932 → Score 19/100
  4. Domain Discriminability (AUC): 0.9490 → Score 10/100
  5. Dataset Compatibility: 30/100
  6. Overlap Prevalence: 0.0% (0 molecules) → Score 30/100

  Overall Transferability Score: 35/100
  DA Success Probability: 20%
  Recommendation: NOT RECOMMENDED
  Confidence: Low

Property: HALF_LIFE
  1. Chemical Similarity: 0.0944 → Score 18/100
  2. Representation Gap (MMD): 0.0392 → Score 96/100
  3. Label Correspondence (r): 0.0468 → Score 4/100
  4. Domain Discriminability (AUC): 0.9518 → Score 9/100
  5. Dataset Compatibility: 40/100
  6. Overlap Prevalence: 0.0% (0 molecules) → Score 30/100

  Overall Transferability Score: 33/100
  DA Success Probability: 20%
  Recommendation: NOT RECOMMENDED
  Confidence: Low

Property: FU
  1. Chemical Similarity: 0.0966 → Score 19/100
  2. Representation Gap (MMD): 0.0583 → 

In [4]:
# Summary Report
print(f"\n\n{'='*100}")
print("TRANSFERABILITY PROFILE SUMMARY")
print(f"{'='*100}\n")

summary_data = []

for prop, profile in transferability_profiles.items():
    row = {
        'Property': prop,
        'Score': profile['overall_score'],
        'DA_Success': f"{profile['da_success_probability']:.0%}",
        'Recommendation': profile['recommendation'],
        'Confidence': profile['confidence'],
    }
    summary_data.append(row)

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))
print()

print("Score Interpretation:")
print("  80-100: Excellent (DA highly likely to succeed)")
print("  70-79:  Good (DA likely to succeed)")
print("  60-69:  Moderate (DA may improve, worth trying)")
print("  50-59:  Marginal (Limited improvement expected)")
print("  0-49:   Poor (DA not recommended)")
print()



TRANSFERABILITY PROFILE SUMMARY

 Property  Score DA_Success  Recommendation Confidence
clearance     35        20% NOT RECOMMENDED        Low
half_life     33        20% NOT RECOMMENDED        Low
       fu     48        20% NOT RECOMMENDED        Low

Score Interpretation:
  80-100: Excellent (DA highly likely to succeed)
  70-79:  Good (DA likely to succeed)
  60-69:  Moderate (DA may improve, worth trying)
  50-59:  Marginal (Limited improvement expected)
  0-49:   Poor (DA not recommended)



In [5]:
# Detailed Recommendations
print(f"\n{'='*100}")
print("DETAILED RECOMMENDATIONS BY PROPERTY")
print(f"{'='*100}\n")

for prop, profile in sorted(transferability_profiles.items(), 
                             key=lambda x: x[1]['overall_score'], 
                             reverse=True):
    print(f"\n{prop.upper()}")
    print(f"  Overall Score: {profile['overall_score']}/100")
    print(f"  Recommendation: {profile['recommendation']}")
    print(f"\n  Dimension Scores:")
    for dim, score in profile['individual_scores'].items():
        if score is not None:
            print(f"    {dim}: {score}/100")
    
    print(f"\n  Key Findings:")
    
    # Identify bottlenecks
    scores_list = [(dim, score) for dim, score in profile['individual_scores'].items() if score is not None]
    scores_list.sort(key=lambda x: x[1])
    
    if scores_list:
        bottleneck = scores_list[0]
        print(f"    - Bottleneck: {bottleneck[0]} (score {bottleneck[1]}/100)")
    
    strength = scores_list[-1]
    print(f"    - Strength: {strength[0]} (score {strength[1]}/100)")
    
    print(f"\n  Next Steps:")
    if profile['recommendation'] == "STRONGLY RECOMMENDED":
        print(f"    1. Proceed with domain adaptation")
        print(f"    2. Use MMD or CORAL loss for distribution alignment")
        print(f"    3. Expect 70-85% success rate")
    elif profile['recommendation'] == "RECOMMENDED":
        print(f"    1. Domain adaptation is viable")
        print(f"    2. Consider ensemble methods for robustness")
        print(f"    3. Validate with held-out target test set")
    elif profile['recommendation'] == "WORTH TRYING":
        print(f"    1. Attempt domain adaptation with caution")
        print(f"    2. Focus on improving the bottleneck dimension")
        print(f"    3. Use conservative hyperparameters")
    elif profile['recommendation'] == "MARGINAL":
        print(f"    1. Domain adaptation may not be beneficial")
        print(f"    2. Consider alternative approaches:")
        print(f"       - Collect more target data")
        print(f"       - Use semi-supervised learning")
        print(f"       - Improve feature engineering")
    else:  # NOT RECOMMENDED
        print(f"    1. Domain adaptation not recommended for this property")
        print(f"    2. Suggested alternatives:")
        print(f"       - Develop target-only model if sufficient data")
        print(f"       - Gather more bridging data")
        print(f"       - Reconsider source-target pairing")


DETAILED RECOMMENDATIONS BY PROPERTY


FU
  Overall Score: 48/100
  Recommendation: NOT RECOMMENDED

  Dimension Scores:
    Chemical Similarity: 19/100
    Representation Gap: 94/100
    Label Correspondence: 69/100
    Domain Discriminability: 13/100
    Dataset Compatibility: 40/100

  Key Findings:
    - Bottleneck: Domain Discriminability (score 13/100)
    - Strength: Representation Gap (score 94/100)

  Next Steps:
    1. Domain adaptation not recommended for this property
    2. Suggested alternatives:
       - Develop target-only model if sufficient data
       - Gather more bridging data
       - Reconsider source-target pairing

CLEARANCE
  Overall Score: 35/100
  Recommendation: NOT RECOMMENDED

  Dimension Scores:
    Chemical Similarity: 18/100
    Representation Gap: 96/100
    Label Correspondence: 19/100
    Domain Discriminability: 10/100
    Dataset Compatibility: 30/100

  Key Findings:
    - Bottleneck: Domain Discriminability (score 10/100)
    - Strength: Repres

In [6]:
# Save results
with open(os.path.join(RESULTS_DIR, 'phase_07_transferability_profile.json'), 'w') as f:
    json.dump(transferability_profiles, f, indent=2)

print("\n\n✓ Results saved to benchmark/results/")
print(f"  - phase_07_transferability_profile.json")
print("\nPhase 7 Complete! Ready for Phase 8: DA Benchmark Comparison")



✓ Results saved to benchmark/results/
  - phase_07_transferability_profile.json

Phase 7 Complete! Ready for Phase 8: DA Benchmark Comparison
